# 16 · The vLLM flags you will tune
Level L3 · Part 2 · core · ~25 min · builds on 15

Time: about 25 min. Step 3 adds a **1 to 2 minute wait** while the CPU server loads its weights. Read the Picture while it loads. The notebook is not frozen.

This lesson starts the pinned vLLM CPU image (`vllm/vllm-openai-cpu:v0.32.0`) in Docker with the flags of a target, then reads back what the engine did. Without Docker, the cells that start or read the server print **recorded output** from a real run of the same image, labelled as such.

## Goal
By the end of this lesson you can set the memory, batch and length flags of a vLLM server for a stated target, read the effective values back from the startup log and `/metrics`, and say why each flag has the value it has.

## Why it matters
- **Without tuning:** the server runs on the engine's defaults, which fit a general case, not your prompts. Length defaults to the model's own maximum: 32,768 tokens for Qwen2.5-0.5B. In the 2 GiB cache of step 1, that holds about 2.7 full-length requests. Sized for 2,048 tokens, the same cache holds 42.
- **With tuning:** each flag is a number with a reason. The startup log and `/metrics` show the value the engine used, so you check the result, not your intent.
- **When you don't need it:** a one-user demo on a laptop. Keep the defaults and read the startup log once.

## The idea
A vLLM server has four dials that decide how much work it takes at once.

- **Memory:** on a GPU, `--gpu-memory-utilization` (default 0.92) sets the share of the device, and the KV cache gets what is left after the weights. `--kv-cache-memory-bytes` sets the cache directly. The CPU image in this course uses that flag.
- **Length:** the longest request accepted, `--max-model-len`.
- **Batch:** how many requests run together, `--max-num-seqs`, and how many tokens one step may process, `--max-num-batched-tokens`.
- **Switches:** prefix caching (lesson 14) and chunked prefill (lesson 12) are on by default.

Think of a restaurant kitchen. Tables are sequences, plates per round are the token budget, and the fridge is the KV cache. Seat more tables than the fridge can feed and the kitchen stalls mid-service.

## Picture
```mermaid
flowchart TD
  T["Target: 4 CPU cores, 4 agents, longest request 1,617 + 199 tokens"] --> M["Memory: 2 GiB for the KV cache"]
  T --> L["Length: max-model-len 2048"]
  T --> B["Batch: max-num-seqs 4, max-num-batched-tokens 2048"]
  M --> K["Engine: 87,296 KV tokens"]
  L --> K
  B --> S["Scheduler: one step holds up to 2,048 tokens"]
  K --> R["Startup log and /metrics: the effective values"]
  S --> R
```

## Step by step
### Step 1: the target and the arithmetic
The target is the Brightside desk on this 4-core lab machine: up to 4 agents at once, and every request must fit. The trace from lesson 07 gives the longest request: a 1,617-token prompt (the 1,500-token policy plus the question) and a 199-token answer. The KV cache stores keys and values for every layer and KV head. Qwen2.5-0.5B has 24 layers, 2 KV heads and head size 64 (its `config.json`). The lab stores values as float32, 4 bytes each. Predict before you start the server.

In [1]:
import json
from pathlib import Path

trace = [json.loads(line) for line in Path("../data/07/support_trace.jsonl").read_text().splitlines()]
longest_in = max(r["prompt_tokens"] for r in trace)
longest_out = max(r["output_tokens"] for r in trace)
need_len = longest_in + longest_out
print(f"trace: {len(trace)} requests; longest prompt {longest_in:,}, longest answer {longest_out} tokens")
print(f"the length flag must allow at least {need_len:,} tokens per request")

LAYERS, KV_HEADS, HEAD_DIM, FLOAT32, BLOCK = 24, 2, 64, 4, 128   # BLOCK: the block size the engine reports (step 4)
per_token = 2 * LAYERS * KV_HEADS * HEAD_DIM * FLOAT32            # keys and values, all layers, all KV heads

def kv_tokens(budget_bytes, bytes_per_token=per_token):
    """Tokens that fit in a KV budget. The engine allocates whole blocks only."""
    return (budget_bytes // (bytes_per_token * BLOCK)) * BLOCK

budget = 2 * 1024**3
print(f"KV bytes per token: {per_token:,}; KV tokens in 2 GiB: {kv_tokens(budget):,}")
print(f"  at 2,048 tokens per request: {kv_tokens(budget) / 2048:.2f} requests")
print(f"  at the model's maximum of 32,768: {kv_tokens(budget) / 32768:.2f} requests")

trace: 30 requests; longest prompt 1,617, longest answer 199 tokens
the length flag must allow at least 1,816 tokens per request
KV bytes per token: 24,576; KV tokens in 2 GiB: 87,296
  at 2,048 tokens per request: 42.62 requests
  at the model's maximum of 32,768: 2.66 requests


### Step 2: what the help text says about each flag
`vllm serve --help=all` is the source of truth for this image. The next cells print the default line of each flag we tune. `(default: None)` means the engine decides the value when it starts, from its config. It does not mean off. The startup log shows what it decided (step 4).

In [2]:
import re

KEEP = ["--max-model-len", "--gpu-memory-utilization", "--kv-cache-memory-bytes", "--max-num-seqs",
        "--max-num-batched-tokens", "--enable-prefix-caching", "--enable-chunked-prefill"]

def flag_summary(text):
    """One line per flag: the default that its help block states."""
    lines, out = text.splitlines(), []
    for flag in KEEP:
        start = next((n for n, l in enumerate(lines) if re.match(rf"  {flag}[ ,]", l)), None)
        if start is None:
            out.append(f"{flag}: not in this image's help")
            continue
        block = [lines[start]]
        for l in lines[start + 1:]:
            if re.match(r"  --[a-z]", l):          # the next flag starts here
                break
            block.append(l)
        stated = [l.strip() for l in block if "(default" in l or "default value of" in l]
        out.append(f"{flag}: " + (" | ".join(stated) or "no default in the help text"))
    return out

The next cell runs the real help command. It needs Docker.

In [3]:
import subprocess
from pathlib import Path

IMAGE = "vllm/vllm-openai-cpu:v0.32.0"          # the pinned CPU image, the same tag as lessons 08 to 15
Path("../data/16").mkdir(parents=True, exist_ok=True)
run = subprocess.run(["docker", "run", "--rm", "--entrypoint", "vllm", IMAGE, "serve", "--help=all"],
                     capture_output=True, text=True, timeout=600)
if run.returncode == 0 and "--max-model-len" in run.stdout:
    Path("../data/16/help.txt").write_text(run.stdout)
    print("\n".join(flag_summary(run.stdout)))
else:
    print("docker run failed:", run.stderr[-300:])

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


**If the help run was skipped**, this cell prints the recorded help text from the same image.

In [4]:
import json
from pathlib import Path

live = Path("../data/16/help.txt")
if live.exists():
    print("source: saved by an earlier run on the pinned image")
    print("\n".join(flag_summary(live.read_text())))
else:
    rec = json.loads(Path("../data/recorded/16.json").read_text())
    print("source:", rec["note"])
    print("\n".join(rec["help"]))

source: saved by an earlier run on the pinned image
--max-model-len: (default: None)
--gpu-memory-utilization: If unspecified, will use the default value of 0.92. | (default: 0.92)
--kv-cache-memory-bytes: (default: None)
--max-num-seqs: `EngineArgs.create_engine_config`. (default: None)
--max-num-batched-tokens: (default: None)
--enable-prefix-caching: Whether to enable prefix caching. (default: None)
--enable-chunked-prefill: `EngineArgs.create_engine_config`. (default: None)


**You should see:** `--gpu-memory-utilization` with "default value of 0.92", and six lines with "(default: None)". We set the byte budget and the three batch and length flags ourselves, so they appear in the startup log's list of non-default arguments.

### Step 3: start the server with the target flags
This cell holds the names and the flags. The key is optional: with `VLLM_API_KEY` unset the server takes calls without one, which is the lab setting. Each flag in `TARGET` has a reason on its line. The port is this lesson's own: 8160.

In [5]:
import os, socket, subprocess, sys, time, atexit
from pathlib import Path
import httpx
sys.path.insert(0, "..")          # the course folder, so tut_env.py can be imported
from tut_env import env

NAME, PORT = "tut-kv-forge-15h-16", 8160          # this lesson's container and port (lesson 08 uses 8081)
BASE = f"http://127.0.0.1:{PORT}"
MODEL, SERVED = "Qwen/Qwen2.5-0.5B-Instruct", "brightside-qwen-0.5b"
KEY = env("VLLM_API_KEY", "")                     # unset: no key; the value is never printed
TARGET = [MODEL, "--served-model-name", SERVED, "--host", "0.0.0.0",
          "--dtype", "float32", "--enforce-eager",                    # CPU settings, as in lesson 08
          "--max-model-len", "2048",                                   # length: the longest trace request fits
          "--max-num-seqs", "4",                                       # batch: one request per core
          "--max-num-batched-tokens", "2048",                          # token budget per step
          "--kv-cache-memory-bytes", str(2 * 1024**3)]                 # memory: 2 GiB of KV cache
Path.home().joinpath(".cache", "huggingface").mkdir(parents=True, exist_ok=True)

def server_up():
    try:
        return httpx.get(f"{BASE}/health", timeout=2).status_code == 200
    except httpx.TransportError:
        return False

print("port", PORT, "| API key set:", bool(KEY), "| server answering:", server_up())

port 8160 | API key set: False | server answering: False


The next cell starts the container and waits for its health route.

In [6]:
subprocess.run(["docker", "rm", "-f", NAME], capture_output=True)    # clear a container left by an interrupted run
with socket.socket() as s:
    if s.connect_ex(("127.0.0.1", PORT)) == 0:
        raise RuntimeError(f"port {PORT} is busy: stop the other process first")
run_env, key_flag = dict(os.environ), []
if KEY:
    run_env["VLLM_API_KEY"] = KEY                # the server reads this variable; the key stays off the command line
    key_flag = ["-e", "VLLM_API_KEY"]
cmd = ["docker", "run", "-d", "--name", NAME, "-p", f"{PORT}:8000", "--shm-size", "4g",
       "-v", f"{Path.home() / '.cache' / 'huggingface'}:/root/.cache/huggingface", *key_flag, IMAGE, *TARGET]
started = subprocess.run(cmd, capture_output=True, text=True, env=run_env, timeout=120)
if started.returncode != 0:
    raise RuntimeError("docker run failed: " + started.stderr[-300:])
atexit.register(lambda: subprocess.run(["docker", "rm", "-f", NAME], capture_output=True))
deadline = time.time() + 480
while time.time() < deadline and not server_up():
    state = subprocess.run(["docker", "inspect", "-f", "{{.State.Running}}", NAME],
                           capture_output=True, text=True).stdout.strip()
    if state != "true":
        logs = subprocess.run(["docker", "logs", "--tail", "20", NAME], capture_output=True, text=True)
        raise RuntimeError("server container stopped:\n" + (logs.stdout + logs.stderr)[-800:])
    time.sleep(2)
if not server_up():
    raise TimeoutError(f"server not healthy after 480 s on port {PORT}")
print("server healthy on port", PORT)

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


**If the server did not start**, the next cells read the recorded startup log and metrics from the same image (`data/recorded/16.json`) and say so.

### Step 4: read the effective values back
The engine reports its own settings in two places: the startup log (the non-default arguments, the config line and the KV cache size) and `/metrics` (the `cache_config_info` series). This cell reads both: from the live server when it runs, from the recording when it does not.

In [7]:
import ast, json, re
from pathlib import Path

def read_engine():
    """The engine's own account: the startup log and the cache metric, live or recorded."""
    if server_up():
        logs = subprocess.run(["docker", "logs", NAME], capture_output=True, text=True)
        info = httpx.get(f"{BASE}/metrics", timeout=30).text
        return logs.stdout + logs.stderr, info, "live server (this run)"
    rec = json.loads(Path("../data/recorded/16.json").read_text())
    return rec["log"], rec["cache_info"], "recorded output: " + rec["note"]

log, info, source = read_engine()
print("source:", source)
args = ast.literal_eval(re.search(r"non-default args: (\{.*\})", log).group(1))
for k in ("max_model_len", "max_num_seqs", "max_num_batched_tokens", "kv_cache_memory_bytes"):
    print(f"set by us    {k}: {args[k]}")

source: recorded output: a live run of this notebook on the course machine (Docker, pinned CPU image, 4 CPU cores, no GPU), 2026-10-22
set by us    max_model_len: 2048
set by us    max_num_seqs: 4
set by us    max_num_batched_tokens: 2048
set by us    kv_cache_memory_bytes: 2147483648


The rest of what the engine reports: the config line, the KV cache size and the metric labels.

In [8]:
for k in ("max_seq_len", "enable_prefix_caching", "enable_chunked_prefill"):
    print(f"startup log  {k}: {re.search(rf'{k}=(\S+?),', log).group(1)}")
print("startup log  ", re.search(r"(?:CPU|GPU) KV cache size: [\d,]+ tokens", log).group(0))
labels = dict(re.findall(r'(\w+)="([^"]*)"', re.search(r"cache_config_info\{(.*?)\}", info).group(1)))
for k in ("gpu_memory_utilization", "enable_prefix_caching", "block_size", "kv_cache_size_tokens", "kv_cache_max_concurrency"):
    print(f"/metrics     {k}: {labels[k]}")

startup log  max_seq_len: 2048
startup log  enable_prefix_caching: True
startup log  enable_chunked_prefill: True
startup log   CPU KV cache size: 87,296 tokens
/metrics     gpu_memory_utilization: 0.92
/metrics     enable_prefix_caching: True
/metrics     block_size: 128
/metrics     kv_cache_size_tokens: 87296
/metrics     kv_cache_max_concurrency: 42.625


**What you should see:** the four values we set come back unchanged. `enable_prefix_caching` and `enable_chunked_prefill` are `True`, though we did not set them. `gpu_memory_utilization` is `0.92` on `/metrics`, yet on this CPU run it does not size the cache: `--kv-cache-memory-bytes` overrides it. The cache holds 87,296 tokens, and the block size is 128 for this model.

### Step 5: check each choice against the target
Each check follows from the target. The cell runs them on the values the engine reports.

In [9]:
predicted = kv_tokens(2 * 1024**3)
engine_tokens = int(labels["kv_cache_size_tokens"])
print(f"step 1 predicted {predicted:,} KV tokens; the engine reports {engine_tokens:,}")
assert predicted == engine_tokens, "the byte count per token does not match the engine"
checks = {
    "length covers the longest trace request": args["max_model_len"] >= need_len,
    "one step can prefill the longest prompt": args["max_num_batched_tokens"] >= longest_in,
    "the KV cache holds all 4 agents at full length": engine_tokens // args["max_model_len"] >= args["max_num_seqs"],
}
for name, ok in checks.items():
    print(("ok    " if ok else "FAIL  ") + name)
assert all(checks.values()), "a flag does not fit the target"

step 1 predicted 87,296 KV tokens; the engine reports 87,296
ok    length covers the longest trace request
ok    one step can prefill the longest prompt
ok    the KV cache holds all 4 agents at full length


### Why each change: memory and length
| Flag | Value | Why |
|---|---|---|
| `--max-model-len` | 2048 | The longest trace request is 1,617 + 199 = 1,816 tokens. 2,048 covers it. The model's maximum, 32,768, would let one request reserve 16 times as much cache. |
| `--kv-cache-memory-bytes` | 2 GiB | The CPU image sizes its cache from this flag. The cache holds far more than the 4 agents need (step 5), so memory is not the limit here. |
| `--gpu-memory-utilization` | not set (0.92) | On a GPU VM it is the first dial to check, since it sets the KV budget. Here the byte flag decides the budget instead. |

### Why each change: batch and switches
| Flag | Value | Why |
|---|---|---|
| `--max-num-seqs` | 4 | The lab has 4 cores. Four requests run together. More would queue behind the cores (lesson 08 uses 4). |
| `--max-num-batched-tokens` | 2048 | It equals the length limit, so a whole 1,617-token policy prompt fits in one step. A smaller budget splits that prompt over steps and raises the time to first token. |
| prefix caching | on (default) | The 1,500-token policy is shared by every request, so reuse pays (lesson 14). |
| chunked prefill | on (default) | Long prompts are split across steps, so running requests keep producing tokens (lesson 12). |

## At work
- **Developers hit these flags most.** Keyword counts in vLLM's GitHub issues (an upper bound, checked 2026-10-21): `KV cache` in about 5,200 issues, `max_model_len` in about 2,500, `gpu_memory_utilization` in about 2,300. → lesson 27 (measure before you change a flag).
- **Standard benchmarks miss the problem.** Anthropic's 2025 postmortem found a routing change that sent short requests to servers built for long ones, which standard benchmarks did not show. Databricks advises measuring on your own deployment. → lesson 27.
- **Preemption is the sign of too much load.** When the KV cache is full, vLLM pauses requests and recomputes them later. Lowering `max-num-seqs` or raising the memory share reduces it. → lesson 33 (preemption on the dashboard).

## Common mistakes
**1. Setting the length below the request.** The server refuses a request longer than `--max-model-len` with a 400 error. This cell sends a 3,000-token prompt to the lab server.

In [10]:
from openai import OpenAI

client = OpenAI(base_url=f"{BASE}/v1", api_key=KEY or "not-needed")   # the lab server takes calls without a key
client.chat.completions.create(model=SERVED, max_tokens=10,
                               messages=[{"role": "user", "content": "refund " * 3000}])   # BadRequestError: longer than 2,048

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


**Fix:** size the prompts to the length the server accepts, or raise `--max-model-len` and check the new KV tokens with the step 1 arithmetic. A limit that is too low rejects requests. A limit that is too high costs cache.

**2. Using the wrong bytes per value.** The check below uses bf16 (2 bytes) on this float32 run. The engine's count does not match.

In [11]:
bf16_guess = kv_tokens(2 * 1024**3, bytes_per_token=2 * LAYERS * KV_HEADS * HEAD_DIM * 2)   # 2 bytes is bf16
assert bf16_guess == engine_tokens, f"the engine reports {engine_tokens:,} tokens; bf16 arithmetic gives {bf16_guess:,}"
# AssertionError: the engine reports 87,296 tokens; bf16 arithmetic gives 174,720

AssertionError: the engine reports 87,296 tokens; bf16 arithmetic gives 174,720

**Fix:** use the dtype the server runs with, and read it from the startup log (`dtype=`), not from the model card.

**3. Reading "(default: None)" as "off".** The help shows `None` for prefix caching, and the engine turns it on. Only the startup log and `/metrics` say what it does.

Stop the lab server now. The Try-it needs no server.

In [12]:
subprocess.run(["docker", "rm", "-f", NAME], capture_output=True)
print("server stopped:", NAME)

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


## Try it
The Brightside desk moves its lab to a machine with half the cache: 1 GiB of KV cache instead of 2 GiB, with the same 2,048-token length limit. Predict how many KV tokens fit and whether 4 agents at full length still fit. Use `kv_tokens` and `per_token` from step 1. Then run the cell.

In [13]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [14]:
tokens = kv_tokens(1 * 1024**3)
print(f"{tokens:,} KV tokens: {tokens / 2048:.2f} requests of 2,048 tokens")
print("4 agents at full length fit:", tokens // 2048 >= 4)
# 43,648 tokens (341 whole blocks of 128): 21.31 full-length requests.
# Halving the budget halves the concurrency, and 4 agents still fit. Only the memory dial changed.

43,648 KV tokens: 21.31 requests of 2,048 tokens
4 agents at full length fit: True


## Recap
- Four dials: memory (the KV budget), length (`--max-model-len`), batch (`--max-num-seqs` and the token budget per step), and two switches that are on by default.
- Set each dial from the target: KV bytes per token times tokens times requests must fit the budget.
- The startup log and `/metrics` report what the engine used. Check them after every change.

**Interview:** "A vLLM server runs out of memory or queues under load. Which flags do you check, and how do you know which values the engine used?" Strong answer: compare the KV tokens and maximum concurrency in the startup log with the longest request and the number of concurrent requests, read the switches from `/metrics`, then benchmark the change on your own traffic (lesson 27).